# AI-generated Video Detection Model — TinyGRU

Model Summary

```
Video URL/Path
      │
      ▼
Uniform Sampling (N=8 frames)
      │
      ▼
EfficientNet-B0  (frame-wise feature extractor)
      │
      ▼
Feature Vector (per frame, 1280-dim)
      │
      ▼
Tiny GRU
      │
      ▼
Binary Classifier (MLP head)
      │
      ▼
AI Generation Possibility (0~1)
```

Link for Dataset

https://modelscope.cn/datasets/MengxueBoBo/MVAD/tree/master/train/fake_fake/indirect/jimeng

## 1. Environment Setup and Library Installation


In [ ]:
# decord: Fast video decoding library (replaced with opencv-python if decord failed)
!pip install -q torch torchvision datasets decord opencv-python scikit-learn tqdm matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 136.1 MB/s eta 0:00:00


In [ ]:
import os
import io
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt

try:
    import decord
    from decord import VideoReader, cpu
    decord.bridge.set_bridge('native')
    USE_DECORD = True
except Exception as e:
    # replace with opencv if decord not installed or load failed
    print(f"decord load failed, replace with opencv: {e}")
    import cv2
    USE_DECORD = False

def seed_all(seed=42):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)
  torch.backends.cudnn.deterministic = True
  torch.backends.cudnn.benchmark = False

seed = 42
seed_all(seed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

## 2. Load Dataset and Check the Dataset Structure

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import glob
import zipfile

DRIVE_ZIP_ROOT = "/content/drive/MyDrive/mvad_subset/_zips"
LOCAL_ZIP_ROOT = "./mvad_subset/_zips"
LOCAL_EXTRACT_ROOT = "./mvad_subset/_extracted"

os.makedirs(LOCAL_ZIP_ROOT, exist_ok=True)
os.makedirs(LOCAL_EXTRACT_ROOT, exist_ok=True)

In [ ]:
!apt-get install -y rsync -q
!rsync -ah --info=progress2 "{DRIVE_ZIP_ROOT}/" "{LOCAL_ZIP_ROOT}/"

Reading package lists...
Building dependency tree...
Reading state information...
rsync is already the newest version (3.2.7-0ubuntu0.22.04.6).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.
         25.66G 100%   68.23MB/s    0:05:58 (xfr#16, to-chk=0/35)


In [ ]:
zip_paths = glob.glob(os.path.join(LOCAL_ZIP_ROOT, "**", "*.zip"), recursive=True)
print(f"{len(zip_paths)} zip files found")
for p in zip_paths:
    print(" -", p)

16 zip files found
 - ./mvad_subset/_zips/train/real_fake/msrvtt/msrvtt_FoleyCrafter.zip
 - ./mvad_subset/_zips/train/real_fake/internvid/internvid_HunYuan.zip
 - ./mvad_subset/_zips/train/real_fake/openvid/openvid_FoleyCrafter.zip
 - ./mvad_subset/_zips/train/fake_fake/Kling2_5/kling_2_5(250_297).zip
 - ./mvad_subset/_zips/train/fake_fake/Kling2_5/kling2_5_Turbo(1_249).zip
 - ./mvad_subset/_zips/train/fake_fake/KlingO1/klingO1_foleycrafter.zip
 - ./mvad_subset/_zips/train/fake_fake/kling2_1/kling2_1(514_719).zip
 - ./mvad_subset/_zips/train/fake_fake/Kling1_6/kling1_6(0_324).zip
 - ./mvad_subset/_zips/train/fake_fake/Sora2/Sora2（1_994）.zip
 - ./mvad_subset/_zips/train/fake_fake/Jimeng/jimeng_foleycrafter.zip
 - ./mvad_subset/_zips/train/fake_real/Humo/Humo(0_1000).zip
 - ./mvad_subset/_zips/test/real_fake/msvd.zip
 - ./mvad_subset/_zips/test/fake_fake/veo3.zip
 - ./mvad_subset/_zips/test/fake_fake/seedance2_0.zip
 - ./mvad_subset/_zips/test/fake_fake/wan2_6_1.zip
 - ./mvad_subset/_zip

In [ ]:
def infer_label(zip_path: str):
    """ real = 0, fake(AI) = 1 """
    lower = zip_path.lower()
    if "real_fake" in lower:
        return 0
    if "fake_fake" in lower or "fake_real" in lower:
        return 1
    return None # incorrect zip path

def infer_type(zip_path: str, zip_root: str):
    """extract first folder name(train/test) starting from zip_root(./mvad_subset/_zips/)"""
    rel_path = os.path.relpath(zip_path, zip_root)
    return rel_path.split(os.sep)[0]  # e.g., "test/fake_fake/seedance2_0.zip" -> "test"

label_of_extracted_dir = {}

for zip_path in zip_paths:
    label = infer_label(zip_path)
    if label is None:
        print(f"[Warning] Incorrect label found at {zip_path}")

    data_type = infer_type(zip_path, LOCAL_ZIP_ROOT)
    dest_name = os.path.splitext(os.path.basename(zip_path))[0]
    dest_dir = os.path.join(LOCAL_EXTRACT_ROOT, data_type, dest_name)

    # 이미 폴더가 있고 비어있지 않으면 압축 해제만 건너뛰고 딕셔너리에는 추가합니다.
    if not (os.path.isdir(dest_dir) and os.listdir(dest_dir)):
        os.makedirs(dest_dir, exist_ok=True)
        try:
            with zipfile.ZipFile(zip_path, "r") as zf:
                zf.extractall(dest_dir)
            print(f"Finished unzipping: {zip_path} -> {dest_dir}")
        except zipfile.BadZipFile:
            print(f"[Error] Corrupted zip file found at {zip_path} ")
            continue

    if label is not None:
        label_of_extracted_dir[dest_dir] = label

print("\nFinal label_of_extracted_dir:")
for d, l in label_of_extracted_dir.items():
    print(f"{d} -> label={l}")

Finished unzipping: ./mvad_subset/_zips/train/real_fake/msrvtt/msrvtt_FoleyCrafter.zip -> ./mvad_subset/_extracted/train/msrvtt_FoleyCrafter
Finished unzipping: ./mvad_subset/_zips/train/real_fake/internvid/internvid_HunYuan.zip -> ./mvad_subset/_extracted/train/internvid_HunYuan
Finished unzipping: ./mvad_subset/_zips/train/real_fake/openvid/openvid_FoleyCrafter.zip -> ./mvad_subset/_extracted/train/openvid_FoleyCrafter
Finished unzipping: ./mvad_subset/_zips/train/fake_fake/Kling2_5/kling_2_5(250_297).zip -> ./mvad_subset/_extracted/train/kling_2_5(250_297)
Finished unzipping: ./mvad_subset/_zips/train/fake_fake/Kling2_5/kling2_5_Turbo(1_249).zip -> ./mvad_subset/_extracted/train/kling2_5_Turbo(1_249)
Finished unzipping: ./mvad_subset/_zips/train/fake_fake/KlingO1/klingO1_foleycrafter.zip -> ./mvad_subset/_extracted/train/klingO1_foleycrafter
Finished unzipping: ./mvad_subset/_zips/train/fake_fake/kling2_1/kling2_1(514_719).zip -> ./mvad_subset/_extracted/train/kling2_1(514_719)
Fini

In [ ]:
import json
import glob
import collections
from datasets import Dataset as HFDataset, DatasetDict

VIDEO_EXTS = (".mp4", ".mov", ".avi", ".mkv", ".webm", ".m4v")

records = []
records_test = []

for dest_dir, label in label_of_extracted_dir.items():
    video_paths = []
    for ext in VIDEO_EXTS:
        video_paths.extend(glob.glob(os.path.join(dest_dir, "**", f"*{ext}"), recursive=True))
    print(f"{os.path.basename(dest_dir)}: {len(video_paths)} Video samples (label={label})")

    rel_dir = os.path.relpath(dest_dir, LOCAL_EXTRACT_ROOT)
    data_type = rel_dir.split(os.sep)[0]  # "train" or "test"

    target_list = records_test if data_type == "test" else records
    for p in video_paths:
        target_list.append({"video": p, "label": label})

BAD_PATHS_FILE = "/content/drive/MyDrive/mvad_subset/bad_video_paths.json"

if os.path.exists(BAD_PATHS_FILE):
    with open(BAD_PATHS_FILE) as f:
        bad_paths_set = set(json.load(f)["bad_paths"])
    print(f"Filtered {len(bad_paths_set)} samples")
else:
    bad_paths_set = set()

records = [r for r in records if r["video"] not in bad_paths_set]
records_test = [r for r in records_test if r["video"] not in bad_paths_set]

print(f"# of Train samples: {len(records)}, # of Test samples: {len(records_test)}")

ds_dict = {"train": HFDataset.from_list(records),
           "test": HFDataset.from_list(records_test)}

ds = DatasetDict(ds_dict)
print(ds)

label_counts = collections.Counter(r["label"] for r in records)
test_label_counts = collections.Counter(r["label"] for r in records_test)

print("Number of train samples by class: ", dict(label_counts))
print("Number of test samples by class: ", dict(test_label_counts))

msrvtt_FoleyCrafter: 2000 Video samples (label=0)
internvid_HunYuan: 1974 Video samples (label=0)
openvid_FoleyCrafter: 3000 Video samples (label=0)
kling_2_5(250_297): 48 Video samples (label=1)
kling2_5_Turbo(1_249): 249 Video samples (label=1)
klingO1_foleycrafter: 1100 Video samples (label=1)
kling2_1(514_719): 206 Video samples (label=1)
kling1_6(0_324): 323 Video samples (label=1)
Sora2（1_994）: 995 Video samples (label=1)
jimeng_foleycrafter: 1187 Video samples (label=1)
Humo(0_1000): 1000 Video samples (label=1)
msvd: 500 Video samples (label=0)
veo3: 40 Video samples (label=1)
seedance2_0: 170 Video samples (label=1)
wan2_6_1: 140 Video samples (label=1)
kling-avatar: 150 Video samples (label=1)
Filtered 9 samples
# of Train samples: 12073, # of Test samples: 1000
DatasetDict({
    train: Dataset({
        features: ['video', 'label'],
        num_rows: 12073
    })
    test: Dataset({
        features: ['video', 'label'],
        num_rows: 1000
    })
})
Number of train sample

## 3. Config [수정됨]

In [ ]:
CONFIG = {
    "video_column": "video",
    "label_column": "label",
    "real_label_values": [0, "real", "Real", False],
    "fake_label_values": [1, "fake", "Fake", True],

    # Sampling Frame
    "num_frames": 8,

    # Model
    "backbone": "efficientnet_b0",
    "freeze_backbone": False,
    "hidden_dim": 64,
    "gru_layers": 1, # Tiny GRU 구조 유지를 위해 1개 레이어 지정
    "dropout": 0.5,

    # Train
    "batch_size": 4,
    "num_epochs": 10,
    "patience": 3,
    "lr": 1e-4,
    "weight_decay": 1e-5,
    "val_ratio": 0.05,
    "num_workers": 4,
    "checkpoint_path": "/content/drive/MyDrive/mvad_model/best_tinygru_b0_frame_8_posweight_1.2_hidden_128_augmentation.pt",
}

BACKBONE_INFO = {
    "efficientnet_b0": {"input_size": 224, "feat_dim": 1280},
}

IMG_SIZE = BACKBONE_INFO[CONFIG["backbone"]]["input_size"] # 224

## 4. Uniform Frame Sampling

Regardless of playback duration, **sample a fixed number (N) of frames** from the video at **evenly spaced intervals.**

* If the total number of frames is greater than or equal to N: sample using the indices from `linspace(0, total_frames-1, N)`
* If the total number of frames is less than N: pad by repeating the last frame until N frames are reached

In [ ]:
def _uniform_indices(total_frames: int, num_frames: int):
    # return sampled frame index
    if total_frames <= 0:
        return [0] * num_frames
    if total_frames >= num_frames:
        idx = np.linspace(0, total_frames - 1, num_frames)
        return np.round(idx).astype(int).tolist()
    else:
        # pad by repeating the last frame
        idx = list(range(total_frames)) + [total_frames - 1] * (num_frames - total_frames)
        return idx


def _resolve_video_source(video_data):
    """ Convert HF datasets' video column values into readable datas types """
    if isinstance(video_data, str):
        return video_data  # file path
    if isinstance(video_data, dict):
        if video_data.get("path"):
            return video_data["path"]
        if video_data.get("bytes"):
            return io.BytesIO(video_data["bytes"])
    if isinstance(video_data, (bytes, bytearray)):
        return io.BytesIO(video_data)
    if hasattr(video_data, "path"): # HF Video object has its own local path attritube
        return video_data.path
    raise ValueError(f"Not supported video type: {type(video_data)}")


def sample_frames_uniform(video_data, num_frames: int):
    """Uniformly samples num_frames frames from the video and returns them as an (N, H, W, 3) uint8 RGB array."""
    source = _resolve_video_source(video_data)

    if USE_DECORD:
        try:
            vr = VideoReader(source if isinstance(source, str) else source, ctx=cpu(0))
            total = len(vr)
            indices = _uniform_indices(total, num_frames)
            frames = vr.get_batch(indices).asnumpy()  # (N, H, W, 3) RGB
            return frames
        except Exception as e:
            print(f"failed to decord ({e}), will be replaced with opencv.")

    # opencv
    if isinstance(source, io.BytesIO):
        import tempfile
        tmp = tempfile.NamedTemporaryFile(suffix=".mp4", delete=False)
        tmp.write(source.getvalue())
        tmp.close()
        cap_path = tmp.name
    else:
        cap_path = source

    cap = cv2.VideoCapture(cap_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = set(_uniform_indices(total, num_frames))
    idx_to_frame = {}
    i = 0
    while cap.isOpened() and len(idx_to_frame) < len(indices):
        ret, frame = cap.read()
        if not ret:
            break
        if i in indices:
            idx_to_frame[i] = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        i += 1
    cap.release()

    ordered_indices = _uniform_indices(total, num_frames)
    frames = []
    last_valid = None
    for idx in ordered_indices:
        if idx in idx_to_frame:
            last_valid = idx_to_frame[idx]
            frames.append(last_valid)
        elif last_valid is not None:
            frames.append(last_valid)
        else:
            # Replace frames with black images if not readable
            frames.append(np.zeros((224, 224, 3), dtype=np.uint8))
    return np.stack(frames, axis=0)


## 5. Defining PyTorch Dataset

In [ ]:
class RandomJPEGCompression:
    def __init__(self, quality_range=(30, 90), p=0.5):
        self.quality_range = quality_range
        self.p = p

    def __call__(self, img):
        if random.random() > self.p:
            return img
        import io
        quality = random.randint(*self.quality_range)
        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=quality)
        buffer.seek(0)
        return Image.open(buffer).convert("RGB")


class RandomGaussianBlur:
    def __init__(self, radius_range=(0.1, 2.0), p=0.3):
        self.radius_range = radius_range
        self.p = p

    def __call__(self, img):
        if random.random() > self.p:
            return img
        from PIL import ImageFilter
        radius = random.uniform(*self.radius_range)
        return img.filter(ImageFilter.GaussianBlur(radius))


class RandomDownUpsample:
    def __init__(self, scale_range=(0.5, 1.0), p=0.3):
        self.scale_range = scale_range
        self.p = p

    def __call__(self, img):
        if random.random() > self.p:
            return img
        w, h = img.size
        scale = random.uniform(*self.scale_range)
        small = img.resize((max(1, int(w*scale)), max(1, int(h*scale))))
        return small.resize((w, h))

In [ ]:
from PIL import Image

def resolve_label(raw_label):
    if raw_label in CONFIG["fake_label_values"]:
        return 1
    if raw_label in CONFIG["real_label_values"]:
        return 0
    if isinstance(raw_label, (int, np.integer)) and raw_label in (0, 1):
        return int(raw_label)
    raise ValueError(f"Unknown label value: {raw_label!r}.")


class MVADFrameDataset(Dataset):
    """Wraps an MVAD split from HF datasets and converts videos into uniformly
    sampled frame tensors.

    If cache_dir is specified, the raw uniformly-sampled frames (uint8, before
    normalization) from each video are cached to disk as .npy files. Video
    decoding (the slowest part) then only happens on the 1st epoch; from the
    2nd epoch onward, frames are simply read from the cache, making things much faster.
    """

    def __init__(self, hf_split, num_frames: int, img_size: int, train: bool = True, cache_dir: str = None):
        self.hf_split = hf_split
        self.num_frames = num_frames
        self.img_size = img_size
        self.train = train
        self.cache_dir = cache_dir
        if self.cache_dir is not None:
            os.makedirs(self.cache_dir, exist_ok=True)

        # EfficientNet(ImageNet Pretrained) Standardization
        mean = [0.485, 0.456, 0.406]
        std = [0.229, 0.224, 0.225]

        if train:
            self.transform = transforms.Compose([
                transforms.ToPILImage(),
                transforms.Resize((img_size, img_size)),
                transforms.RandomHorizontalFlip(p=0.5),
                RandomJPEGCompression(p=0.5),
                RandomGaussianBlur(p=0.3),
                RandomDownUpsample(p=0.3),
                transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
                transforms.ToTensor(),
                transforms.Normalize(mean=mean, std=std),
            ])
        else:
            self.transform = transforms.Compose([
                transforms.ToPILImage(),
                transforms.Resize((img_size, img_size)),
                transforms.ToTensor(),
                transforms.Normalize(mean=mean, std=std),
            ])

    def __len__(self):
        return len(self.hf_split)

    def _load_frames(self, idx, video_data):
        if self.cache_dir is not None:
            cache_path = os.path.join(self.cache_dir, f"{idx}_{self.num_frames}f_{self.img_size}px.npy")
            if os.path.isfile(cache_path):
                return np.load(cache_path)

        try:
            frames = sample_frames_uniform(video_data, self.num_frames)  # (N, H, W, 3) uint8, original resolution
        except Exception as e:
            print(f"[Warning] index={idx} video loading failed {e}.")
            frames = np.zeros((self.num_frames, self.img_size, self.img_size, 3), dtype=np.uint8)

        if self.cache_dir is not None:
            resized = np.stack([
                np.array(Image.fromarray(f).resize((self.img_size, self.img_size)))
                for f in frames
            ])
            cache_path = os.path.join(self.cache_dir, f"{idx}_{self.num_frames}f_{self.img_size}px.npy")
            np.save(cache_path, resized)
            return resized

        return frames

    def __getitem__(self, idx):
        example = self.hf_split[idx]
        video_data = example[CONFIG["video_column"]]
        label = resolve_label(example[CONFIG["label_column"]])

        frames = self._load_frames(idx, video_data)
        frame_tensors = torch.stack([self.transform(f) for f in frames], dim=0)  # (N, C, H, W)
        return frame_tensors, torch.tensor(label, dtype=torch.float32)


## 6. Model Definition

This implements the `Frame -> EfficientNet -> Feature Vector -> Mean Pooling -> Binary Classifier` structure as described.

- Input tensor shape: `(B, T, C, H, W)` (B = batch size, T = number of frames)
- Reshape `(B, T, C, H, W) -> (B*T, C, H, W)`, then pass through EfficientNet in a single forward pass to extract per-frame feature vectors
- Restore `(B*T, feat_dim) -> (B, T, feat_dim)`, then apply mean pooling over the T (frame) axis
- Pass the pooled feature through an MLP binary classifier head to output a logit

In [ ]:
class EfficientNetTinyGRUClassifier(nn.Module):
    def __init__(self, backbone_name="efficientnet_b0", hidden_dim=64, gru_layers=1, dropout=0.5, pretrained=True, freeze_backbone=False):
        super().__init__()

        weights = EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        model = efficientnet_b0(weights=weights)

        feat_dim = model.classifier[1].in_features  # EfficientNet-B0의 출력 차원: 1280
        self.features = model.features
        self.avgpool = model.avgpool
        self.feat_dim = feat_dim

        if freeze_backbone:
            for p in self.features.parameters():
                p.requires_grad = False

        # Tiny GRU layer
        self.gru = nn.GRU(
            input_size=feat_dim,
            hidden_size=hidden_dim,
            num_layers=gru_layers,
            batch_first=True
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        B, T, C, H, W = x.shape

        x = x.view(B * T, C, H, W)
        feats = self.features(x)            # (B*T, 1280, 7, 7)
        feats = self.avgpool(feats)         # (B*T, 1280, 1, 1)
        feats = torch.flatten(feats, 1)     # (B*T, 1280)

        feats = feats.view(B, T, self.feat_dim)  # (B, T, 1280)

        # out: 각 타임스텝(프레임)별 GRU 출력 결과 -> (B, T, hidden_dim)
        # _h: 마지막 레이어의 최종 hidden state -> (gru_layers, B, hidden_dim)
        out, _h = self.gru(feats)

        # 시계열의 마지막 프레임이 가진 정보(가장 최신 은닉 상태)
        last_temporal_feature = out[:, -1, :]  (B, hidden_dim)

        logit = self.classifier(last_temporal_feature).squeeze(1)  # (B,)
        return logit

model = EfficientNetTinyGRUClassifier(
    backbone_name=CONFIG["backbone"],
    hidden_dim=CONFIG["hidden_dim"],
    gru_layers=CONFIG["gru_layers"],
    dropout=CONFIG["dropout"],
    pretrained=True,
    freeze_backbone=CONFIG["freeze_backbone"],
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Parameters: {n_params:,} | Trainable Parameters: {n_trainable:,}")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 211MB/s]


Total Parameters: 4,270,205 | Trainable Parameters: 4,270,205


## 7. DataLoader

In [ ]:
records = []
records_test = []

train_hf = ds["train"]
test_hf = ds["test"]

FRAME_CACHE_ROOT = "./frame_cache"
train_dataset = MVADFrameDataset(
    train_hf, CONFIG["num_frames"], IMG_SIZE, train=True,
    cache_dir=os.path.join(FRAME_CACHE_ROOT, "train"),
)

test_dataset = MVADFrameDataset(
    test_hf, CONFIG["num_frames"], IMG_SIZE, train=False,
    cache_dir=os.path.join(FRAME_CACHE_ROOT, "test"),
)

val_dataset_no_aug = MVADFrameDataset(
    train_hf, CONFIG["num_frames"], IMG_SIZE, train=False,
    cache_dir=os.path.join(FRAME_CACHE_ROOT, "train"),
)

val_size = int(len(train_dataset) * CONFIG["val_ratio"])
train_size = len(train_dataset) - val_size
train_subset, val_subset = random_split(
    train_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(seed),
)

train_dataset = torch.utils.data.Subset(train_dataset, train_subset.indices)
val_dataset = torch.utils.data.Subset(val_dataset_no_aug, val_subset.indices)

print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}")

train_loader = DataLoader(
    train_dataset, batch_size=CONFIG["batch_size"], shuffle=True,
    num_workers=0, pin_memory=False, drop_last=True,
)

val_loader = DataLoader(
    val_dataset, batch_size=CONFIG["batch_size"], shuffle=False,
    num_workers=CONFIG["num_workers"], pin_memory=False,
)

test_loader = DataLoader(
    test_dataset, batch_size=CONFIG["batch_size"], shuffle=False,
    num_workers=CONFIG["num_workers"], pin_memory=False,
)

Train size: 11470, Val size: 603


## 8. Loss Function and Optimizer

In [ ]:
# Class-Balanced Weighting
if "label_counts" in dir():
    n_real = label_counts.get(0, 0)
    n_fake = label_counts.get(1, 0)
    if n_fake > 0:
      # Scale up pos_weight to suppress FN - case2
        pos_weight_value = (n_real / n_fake)*1.2
        pos_weight = torch.tensor([pos_weight_value], device=DEVICE)
        print(f"class-balanced weight={pos_weight_value:.3f}")
    else:
        pos_weight = None
else:
    pos_weight = None

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"],
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["num_epochs"])

class-balanced weight=1.636


## 9. Model Train and Validation

In [ ]:
USE_AMP = DEVICE.type == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

def run_epoch(model, loader, optimizer=None, train=True):
    model.train() if train else model.eval()

    total_loss = 0.0
    all_labels, all_probs = [], []

    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        pbar = tqdm(loader, desc="Train" if train else "Val")
        for frames, labels in pbar:
            frames = frames.to(DEVICE, non_blocking=True)   # (B, T, C, H, W)
            labels = labels.to(DEVICE, non_blocking=True)   # (B,)

            with torch.autocast(device_type="cuda", enabled=USE_AMP):
                logits = model(frames)
                loss = criterion(logits, labels)

            if train:
                optimizer.zero_grad()
                if USE_AMP:
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    optimizer.step()

            total_loss += loss.item() * frames.size(0)
            probs = torch.sigmoid(logits).detach().cpu().numpy()
            all_probs.extend(probs.tolist())
            all_labels.extend(labels.detach().cpu().numpy().tolist())

            pbar.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(loader.dataset)
    preds = [1 if p >= 0.5 else 0 for p in all_probs]
    acc = accuracy_score(all_labels, preds)
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        auc = float("nan")  # e.g., when a batch contains only one class
    f1 = f1_score(all_labels, preds, zero_division=0)

    return avg_loss, acc, auc, f1, all_labels, all_probs

/tmp/ipykernel_4680/495532786.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


In [ ]:
history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_auc": [], "val_f1": []}
best_val_f1 = -1.0
best_val_loss = float("inf")
epochs_no_improve = 0

for epoch in range(1, CONFIG["num_epochs"] + 1):
    print(f"\nEpoch {epoch}/{CONFIG['num_epochs']}")

    train_loss, train_acc, train_auc, train_f1, _, _ = run_epoch(model, train_loader, optimizer, train=True)
    val_loss, val_acc, val_auc, val_f1, val_labels, val_probs = run_epoch(model, val_loader, train=False)

    scheduler.step()

    print(f"Train  | loss={train_loss:.4f} acc={train_acc:.4f} auc={train_auc:.4f} f1={train_f1:.4f}")
    print(f"Val    | loss={val_loss:.4f} acc={val_acc:.4f} auc={val_auc:.4f} f1={val_f1:.4f}")

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_auc"].append(val_auc)
    history["val_f1"].append(val_f1)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "config": CONFIG,
            "val_f1": val_f1,
        }, CONFIG["checkpoint_path"])
        print(f"  -> Best Performance Saved (val_f1 = {val_f1:.4f})")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        print(f"  (val_f1 not improved: {epochs_no_improve}/{CONFIG['patience']})")

    if epochs_no_improve >= CONFIG["patience"]:
        print(f"\nEarly Stopped.")
        break


print(f"\nFinish. Best Val F1: {best_val_f1:.4f}")



Epoch 1/10


Train:   0%|          | 0/2867 [00:00<?, ?it/s]

Val:   0%|          | 0/151 [00:00<?, ?it/s]

Train  | loss=0.4172 acc=0.8582 auc=0.9350 f1=0.8417
Val    | loss=0.1413 acc=0.9486 auc=0.9943 f1=0.9423
  -> Best Performance Saved (val_f1 = 0.9423)

Epoch 2/10


Train:   0%|          | 0/2867 [00:00<?, ?it/s]

Val:   0%|          | 0/151 [00:00<?, ?it/s]

Train  | loss=0.1722 acc=0.9543 auc=0.9870 f1=0.9464
Val    | loss=0.1053 acc=0.9685 auc=0.9937 f1=0.9639
  -> Best Performance Saved (val_f1 = 0.9639)

Epoch 3/10


Train:   0%|          | 0/2867 [00:00<?, ?it/s]

## 10. Loss Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

axes[0].plot(history["train_loss"], label="train_loss")
axes[0].plot(history["val_loss"], label="val_loss")
axes[0].set_title("Loss Curve")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history["val_auc"], label="val_auc", color="orange")
axes[1].plot(history["val_f1"], label="val_f1", color="red")
axes[1].set_title("Validation AUC / F1")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()


## 11. Validation Confusion Matrix

In [ ]:
val_preds = [1 if p >= 0.5 else 0 for p in val_probs]
cm = confusion_matrix(val_labels, val_preds)

fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Real", "Fake"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["Real", "Fake"])
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="black")
ax.set_title("Confusion Matrix (Validation)")
plt.colorbar(im)
plt.show()


## 14. Model Test

In [ ]:
checkpoint = torch.load(CONFIG["checkpoint_path"], map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
print(f"Best Performance Model Loaded (best at {checkpoint['epoch']} epoch, {checkpoint['val_f1']:.4f} val_f1)")

test_loss, test_acc, test_auc, test_f1, test_labels, test_probs = run_epoch(model, test_loader, train=False)
print(f"\nTest Result | loss={test_loss:.4f} acc={test_acc:.4f} auc={test_auc:.4f} f1={test_f1:.4f}")

In [ ]:
# Test Confusion Matrix
test_preds = [1 if p >= 0.5 else 0 for p in test_probs]
cm_test = confusion_matrix(test_labels, test_preds)

fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(cm_test, cmap="Purples")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Real", "Fake"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["Real", "Fake"])
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm_test[i, j], ha="center", va="center", color="black")
ax.set_title("Confusion Matrix (Test)")
plt.colorbar(im)
plt.show()
